In [3]:
 
# ── 1. IMPORTS ──────────────────────────────────────────────
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import re
import pickle
import warnings
warnings.filterwarnings("ignore")
 
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, f1_score,
                              precision_recall_curve, roc_auc_score,
                              accuracy_score, confusion_matrix)
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.preprocessing import StandardScaler
 
from imblearn.combine import SMOTETomek
import lightgbm as lgb
from transformers import (RobertaTokenizer,
                           RobertaForSequenceClassification,
                           get_linear_schedule_with_warmup)
from tqdm import tqdm
 
print("✅ All libraries loaded!")

✅ All libraries loaded!


In [2]:
!pip install lightgbm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 2.9 MB/s  0:00:083.1 MB/s eta 0:00:01:01


In [4]:
# ── 2. CONFIG ────────────────────────────────────────────────
class CFG:
    # Data
    DATA_PATH       = "fake_job_postings.csv"
    TEST_SIZE       = 0.2
    RANDOM_STATE    = 42
 
    # DistilRoBERTa Fine-tuning
    MODEL_NAME      = "distilroberta-base"
    MAX_LENGTH      = 128
    BATCH_SIZE      = 8          # safe for RTX 3050 4GB VRAM
    EPOCHS          = 3          # 3 epochs is enough for fine-tuning
    LR              = 2e-5
    WARMUP_RATIO    = 0.1
    GRAD_ACCUM      = 4          # effective batch = 8×4 = 32
 
    # TF-IDF
    TFIDF_WORD_FEATURES = 10000
    TFIDF_CHAR_FEATURES = 5000
 
    # Feature Selection
    K_BEST          = 5000
 
    # LightGBM
    LGB_PARAMS = {
        "n_estimators"    : 500,
        "learning_rate"   : 0.05,
        "num_leaves"      : 63,
        "max_depth"       : -1,
        "subsample"       : 0.8,
        "colsample_bytree": 0.8,
        "min_child_samples": 20,
        "reg_alpha"       : 0.1,
        "reg_lambda"      : 0.1,
        "random_state"    : 42,
        "n_jobs"          : -1,
        "verbose"         : -1
    }
 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

✅ Device: cuda
   GPU: NVIDIA GeForce RTX 3050
   VRAM: 8.2 GB


In [5]:
data = pd.read_csv(CFG.DATA_PATH)
print(f"\n✅ Dataset loaded: {data.shape}")
print(f"   Fraud ratio: {data['fraudulent'].mean():.3f} "
      f"({data['fraudulent'].sum()} fraud / {len(data)} total)")


✅ Dataset loaded: (17880, 18)
   Fraud ratio: 0.048 (866 fraud / 17880 total)


In [6]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"[^a-zA-Z ]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text
 
text = (
    data["title"].fillna("") + " " +
    data["company_profile"].fillna("") + " " +
    data["description"].fillna("") + " " +
    data["requirements"].fillna("") + " " +
    data["benefits"].fillna("")
)
text  = text.apply(clean_text)
labels = data["fraudulent"].values

In [7]:
def extract_metadata(df):
    meta = pd.DataFrame()
 
    meta["has_logo"]          = df["has_company_logo"].fillna(0).astype(int)
    meta["has_questions"]     = df["has_questions"].fillna(0).astype(int)
    meta["has_salary"]        = (~df["salary_range"].isna()).astype(int)
    meta["has_experience"]    = (~df["required_experience"].isna()).astype(int)
    meta["has_education"]     = (~df["required_education"].isna()).astype(int)
 
    emp = df["employment_type"].fillna("Unknown")
    for cat in ["Full-time", "Part-time", "Contract", "Temporary"]:
        meta[f"emp_{cat.lower().replace('-','_')}"] = (emp == cat).astype(int)
 
    meta["desc_len"]          = df["description"].fillna("").apply(len)
    meta["profile_len"]       = df["company_profile"].fillna("").apply(len)
    meta["req_len"]           = df["requirements"].fillna("").apply(len)
    meta["very_short_desc"]   = (meta["desc_len"] < 100).astype(int)
    meta["no_company_profile"]= (meta["profile_len"] == 0).astype(int)
 
    # Normalize length features
    for col in ["desc_len", "profile_len", "req_len"]:
        meta[col] = (meta[col] - meta[col].mean()) / (meta[col].std() + 1e-8)
 
    return meta.values.astype(float)
 
metadata = extract_metadata(data)
print(f"✅ Metadata features: {metadata.shape[1]}")

✅ Metadata features: 14


In [9]:
(X_train_text, X_test_text,
 y_train, y_test,
 meta_train, meta_test) = train_test_split(
    text.values, labels, metadata,
    test_size=CFG.TEST_SIZE,
    stratify=labels,
    random_state=CFG.RANDOM_STATE
)
print(f"✅ Train: {len(X_train_text)} | Test: {len(X_test_text)}")

✅ Train: 14304 | Test: 3576


In [10]:
print("\n⚙️  Building TF-IDF features...")
 
tfidf_word = TfidfVectorizer(
    max_features=CFG.TFIDF_WORD_FEATURES,
    ngram_range=(1, 3),
    stop_words="english",
    min_df=2,
    sublinear_tf=True
)
X_train_tfidf_w = tfidf_word.fit_transform(X_train_text).toarray()
X_test_tfidf_w  = tfidf_word.transform(X_test_text).toarray()
 
tfidf_char = TfidfVectorizer(
    max_features=CFG.TFIDF_CHAR_FEATURES,
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=3,
    sublinear_tf=True
)
X_train_tfidf_c = tfidf_char.fit_transform(X_train_text).toarray()
X_test_tfidf_c  = tfidf_char.transform(X_test_text).toarray()
 
X_train_tfidf = np.hstack((X_train_tfidf_w, X_train_tfidf_c))
X_test_tfidf  = np.hstack((X_test_tfidf_w,  X_test_tfidf_c))
print(f"✅ TF-IDF shape: {X_train_tfidf.shape}")


⚙️  Building TF-IDF features...
✅ TF-IDF shape: (14304, 15000)


In [11]:
print(f"\n⚙️  Loading {CFG.MODEL_NAME}...")
 
tokenizer     = RobertaTokenizer.from_pretrained(CFG.MODEL_NAME)
roberta_model = RobertaForSequenceClassification.from_pretrained(
    CFG.MODEL_NAME,
    num_labels=2
).to(device)
 
# ── Dataset class ────────────────────────────────────────────
class JobDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.texts  = texts
        self.labels = labels
 
    def __len__(self):
        return len(self.texts)
 
    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            max_length=CFG.MAX_LENGTH,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item
 
train_dataset = JobDataset(X_train_text, y_train)
test_dataset  = JobDataset(X_test_text)
 
train_loader  = DataLoader(train_dataset, batch_size=CFG.BATCH_SIZE,
                            shuffle=True,  num_workers=2, pin_memory=True)
test_loader   = DataLoader(test_dataset,  batch_size=CFG.BATCH_SIZE,
                            shuffle=False, num_workers=2, pin_memory=True)
 
# ── Fine-tuning loop ─────────────────────────────────────────
total_steps   = (len(train_loader) // CFG.GRAD_ACCUM) * CFG.EPOCHS
warmup_steps  = int(total_steps * CFG.WARMUP_RATIO)
 
optimizer  = optim.AdamW(roberta_model.parameters(), lr=CFG.LR, weight_decay=0.01)
scheduler  = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
 
# Class weights for imbalanced data
fraud_weight  = len(y_train) / (2 * y_train.sum())
class_weights = torch.tensor([1.0, fraud_weight], dtype=torch.float).to(device)
loss_fn       = nn.CrossEntropyLoss(weight=class_weights)
 
print(f"✅ Fine-tuning for {CFG.EPOCHS} epochs "
      f"(effective batch={CFG.BATCH_SIZE * CFG.GRAD_ACCUM})...")
 
best_loss = float("inf")
for epoch in range(CFG.EPOCHS):
    roberta_model.train()
    total_loss = 0
    optimizer.zero_grad()
 
    for step, batch in enumerate(tqdm(train_loader,
                                       desc=f"Epoch {epoch+1}/{CFG.EPOCHS}")):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels_batch   = batch["labels"].to(device)
 
        outputs = roberta_model(input_ids=input_ids,
                                 attention_mask=attention_mask)
        loss    = loss_fn(outputs.logits, labels_batch)
        loss    = loss / CFG.GRAD_ACCUM
        loss.backward()
 
        total_loss += loss.item() * CFG.GRAD_ACCUM
 
        if (step + 1) % CFG.GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(roberta_model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
 
    avg_loss = total_loss / len(train_loader)
    print(f"   Epoch {epoch+1} loss: {avg_loss:.4f}")
 
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(roberta_model.state_dict(), "best_roberta.pt")
 
# Load best weights
roberta_model.load_state_dict(torch.load("best_roberta.pt"))
print("✅ Fine-tuning complete! Best model loaded.")
 


⚙️  Loading distilroberta-base...


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

   Epoch 1 loss: 0.3612


Epoch 2/3: 100%|████████████████████████████| 1788/1788 [02:21<00:00, 12.67it/s]


   Epoch 2 loss: 0.1979


Epoch 3/3: 100%|████████████████████████████| 1788/1788 [02:23<00:00, 12.48it/s]


   Epoch 3 loss: 0.1201
✅ Fine-tuning complete! Best model loaded.


In [12]:
def extract_embeddings(loader, desc="Extracting"):
    roberta_model.eval()
    all_embeddings = []
 
    with torch.no_grad():
        for batch in tqdm(loader, desc=desc):
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
 
            outputs = roberta_model.roberta(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            # CLS token
            cls = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            all_embeddings.append(cls)
 
    return np.vstack(all_embeddings)
 
print("\n⚙️  Extracting fine-tuned embeddings...")
train_embeddings = extract_embeddings(train_loader, "Train embeddings")
test_embeddings  = extract_embeddings(test_loader,  "Test embeddings")
print(f"✅ Embeddings shape: {train_embeddings.shape}")


⚙️  Extracting fine-tuned embeddings...


Test embeddings: 100%|████████████████████████| 447/447 [00:09<00:00, 45.73it/s]

✅ Embeddings shape: (14304, 768)


In [13]:
class AttentionFusion(nn.Module):
    """
    Learns importance weights for each feature group:
    [DistilRoBERTa | TF-IDF | Metadata]
    Weighted sum → fused representation
    """
    def __init__(self, dims):
        super().__init__()
        # One scalar weight per feature group
        self.weights = nn.Parameter(torch.ones(len(dims)))
        self.dims    = dims
 
    def forward(self, features):
        # Softmax ensures weights sum to 1
        w = torch.softmax(self.weights, dim=0)
        # Scale each group by its learned weight
        out = []
        for i, f in enumerate(features):
            out.append(f * w[i].item())
        return np.hstack(out)
 
# Convert to tensors for weight learning
print("\n⚙️  Learning attention fusion weights...")
 
# Normalize each feature group to same scale
scaler_bert = StandardScaler()
scaler_tfidf = StandardScaler()
scaler_meta  = StandardScaler()
 
train_bert_n  = scaler_bert.fit_transform(train_embeddings)
test_bert_n   = scaler_bert.transform(test_embeddings)
 
train_tfidf_n = scaler_tfidf.fit_transform(X_train_tfidf)
test_tfidf_n  = scaler_tfidf.transform(X_test_tfidf)
 
train_meta_n  = scaler_meta.fit_transform(meta_train)
test_meta_n   = scaler_meta.transform(meta_test)
 
# Train attention weights via small neural net
class FusionNet(nn.Module):
    def __init__(self, bert_dim, tfidf_dim, meta_dim):
        super().__init__()
        self.bert_weight  = nn.Parameter(torch.tensor(1.0))
        self.tfidf_weight = nn.Parameter(torch.tensor(1.0))
        self.meta_weight  = nn.Parameter(torch.tensor(1.0))
 
        total_dim = bert_dim + tfidf_dim + meta_dim
        self.classifier = nn.Sequential(
            nn.Linear(total_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 2)
        )
 
    def forward(self, bert, tfidf, meta):
        w = torch.softmax(
            torch.stack([self.bert_weight,
                         self.tfidf_weight,
                         self.meta_weight]), dim=0
        )
        fused = torch.cat([bert  * w[0],
                           tfidf * w[1],
                           meta  * w[2]], dim=1)
        return self.classifier(fused), w
 
# Quick training to learn fusion weights
fusion_net = FusionNet(
    bert_dim  = train_bert_n.shape[1],
    tfidf_dim = train_tfidf_n.shape[1],
    meta_dim  = train_meta_n.shape[1]
).to(device)
 
# Convert to tensors
T = lambda x: torch.tensor(x, dtype=torch.float32)
train_bert_t  = T(train_bert_n)
train_tfidf_t = T(train_tfidf_n)
train_meta_t  = T(train_meta_n)
train_y_t     = torch.tensor(y_train, dtype=torch.long)
 
fusion_dataset = torch.utils.data.TensorDataset(
    train_bert_t, train_tfidf_t, train_meta_t, train_y_t
)
fusion_loader  = DataLoader(fusion_dataset, batch_size=64, shuffle=True)
 
fusion_opt = optim.Adam(fusion_net.parameters(), lr=1e-3)
fusion_loss_fn = nn.CrossEntropyLoss(
    weight=torch.tensor([1.0, fraud_weight], dtype=torch.float).to(device)
)
 
print("⚙️  Training attention fusion (5 epochs)...")
for epoch in range(5):
    fusion_net.train()
    for b_bert, b_tfidf, b_meta, b_y in fusion_loader:
        b_bert  = b_bert.to(device)
        b_tfidf = b_tfidf.to(device)
        b_meta  = b_meta.to(device)
        b_y     = b_y.to(device)
 
        logits, weights = fusion_net(b_bert, b_tfidf, b_meta)
        loss = fusion_loss_fn(logits, b_y)
 
        fusion_opt.zero_grad()
        loss.backward()
        fusion_opt.step()
 
# Extract learned weights
fusion_net.eval()
with torch.no_grad():
    _, learned_weights = fusion_net(
        T(train_bert_n[:1]).to(device),
        T(train_tfidf_n[:1]).to(device),
        T(train_meta_n[:1]).to(device)
    )
    w = learned_weights.cpu().numpy()
 
print(f"\n✅ Learned attention weights:")
print(f"   DistilRoBERTa : {w[0]:.4f}")
print(f"   TF-IDF        : {w[1]:.4f}")
print(f"   Metadata      : {w[2]:.4f}")
 
# Apply learned weights to features
X_train_fused = np.hstack([
    train_bert_n  * w[0],
    train_tfidf_n * w[1],
    train_meta_n  * w[2]
])
X_test_fused = np.hstack([
    test_bert_n  * w[0],
    test_tfidf_n * w[1],
    test_meta_n  * w[2]
])
print(f"✅ Fused feature shape: {X_train_fused.shape}")


⚙️  Learning attention fusion weights...
⚙️  Training attention fusion (5 epochs)...

✅ Learned attention weights:
   DistilRoBERTa : 0.2901
   TF-IDF        : 0.3381
   Metadata      : 0.3718
✅ Fused feature shape: (14304, 15782)


In [14]:
print(f"\n⚙️  Feature selection (k={CFG.K_BEST})...")
selector    = SelectKBest(mutual_info_classif, k=CFG.K_BEST)
X_train_sel = selector.fit_transform(X_train_fused, y_train)
X_test_sel  = selector.transform(X_test_fused)
print(f"✅ After selection: {X_train_sel.shape}")


⚙️  Feature selection (k=5000)...
✅ After selection: (14304, 5000)


In [15]:
print("\n⚙️  Applying SMOTE-Tomek...")
smote_tomek = SMOTETomek(random_state=CFG.RANDOM_STATE)
X_train_bal, y_train_bal = smote_tomek.fit_resample(X_train_sel, y_train)
print(f"✅ Balanced shape: {X_train_bal.shape}")
print(f"   Class dist: {np.bincount(y_train_bal)}")
 


⚙️  Applying SMOTE-Tomek...
✅ Balanced shape: (27222, 5000)
   Class dist: [13611 13611]


In [16]:
print("\n⚙️  Training LightGBM...")
lgb_model = lgb.LGBMClassifier(**CFG.LGB_PARAMS)
lgb_model.fit(
    X_train_bal, y_train_bal,
    eval_set=[(X_test_sel, y_test)],
    callbacks=[lgb.early_stopping(50, verbose=False),
               lgb.log_evaluation(100)]
)
print("✅ LightGBM training complete!")


⚙️  Training LightGBM...
[100]	valid_0's binary_logloss: 0.0510596
[200]	valid_0's binary_logloss: 0.0439878
✅ LightGBM training complete!


In [17]:
probs = lgb_model.predict_proba(X_test_sel)[:, 1]
 
precision, recall, thresholds = precision_recall_curve(y_test, probs)
f1_scores     = 2 * precision * recall / (precision + recall + 1e-8)
best_idx       = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
print(f"\n🎯 Best threshold: {best_threshold:.4f} "
      f"(F1={f1_scores[best_idx]:.4f})")
 
predictions = (probs > best_threshold).astype(int)


🎯 Best threshold: 0.3792 (F1=0.8527)


In [18]:
pickle.dump(lgb_model,      open("etff_lgb_model.pkl",   "wb"))
pickle.dump(tfidf_word,     open("etff_tfidf_word.pkl",  "wb"))
pickle.dump(tfidf_char,     open("etff_tfidf_char.pkl",  "wb"))
pickle.dump(selector,       open("etff_selector.pkl",    "wb"))
pickle.dump(best_threshold, open("etff_threshold.pkl",   "wb"))
pickle.dump(scaler_bert,    open("etff_scaler_bert.pkl", "wb"))
pickle.dump(scaler_tfidf,   open("etff_scaler_tfidf.pkl","wb"))
pickle.dump(scaler_meta,    open("etff_scaler_meta.pkl", "wb"))
pickle.dump(w,              open("etff_attn_weights.pkl","wb"))
torch.save(roberta_model.state_dict(), "etff_roberta.pt")
torch.save(fusion_net.state_dict(),    "etff_fusion_net.pt")
 
print("\n✅ All artifacts saved!")
print("   → etff_lgb_model.pkl")
print("   → etff_roberta.pt")
print("   → etff_fusion_net.pt")
print("   → etff_tfidf_word.pkl / etff_tfidf_char.pkl")
print("   → etff_selector.pkl / etff_threshold.pkl")
print("   → etff_scaler_*.pkl")
print("   → etff_attn_weights.pkl")
print("\n🎉 ETFF-Net training complete! Ready for IEEE submission.")
 


✅ All artifacts saved!
   → etff_lgb_model.pkl
   → etff_roberta.pt
   → etff_fusion_net.pt
   → etff_tfidf_word.pkl / etff_tfidf_char.pkl
   → etff_selector.pkl / etff_threshold.pkl
   → etff_scaler_*.pkl
   → etff_attn_weights.pkl

🎉 ETFF-Net training complete! Ready for IEEE submission.


In [19]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, roc_auc_score

# Accuracy
acc = accuracy_score(y_test, predictions)

# Other metrics (recommended)
f1  = f1_score(y_test, predictions)
roc = roc_auc_score(y_test, probs)
cm  = confusion_matrix(y_test, predictions)

print("\n===== MODEL PERFORMANCE =====")
print("Accuracy:", acc)
print("F1 Score:", f1)
print("ROC-AUC:", roc)

print("\nClassification Report:")
print(classification_report(y_test, predictions))

print("\nConfusion Matrix:")
print(cm)


===== MODEL PERFORMANCE =====
Accuracy: 0.9865771812080537
F1 Score: 0.8490566037735849
ROC-AUC: 0.9896334244350871

Classification Report:
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      3403
           1       0.93      0.78      0.85       173

    accuracy                           0.99      3576
   macro avg       0.96      0.89      0.92      3576
weighted avg       0.99      0.99      0.99      3576


Confusion Matrix:
[[3393   10]
 [  38  135]]
